In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:
live = pd.read_csv("../data/samples/mega/live_metrics.csv")
verbose = pd.read_csv("../data/samples/mega/verbose_statements.csv")
initial = pd.read_csv("../data/samples/mega/initial_statement.csv")

In [3]:
live = live.drop(['Unnamed: 0', "_id", "Category", "Collect", ], axis=1)

In [4]:
live['Current Time'] = pd.to_datetime(live['Current Time'])
live['Current Time'] = pd.to_numeric(live['Current Time'])

In [5]:
response_time = verbose['total_duration'].tolist()
eval_rate = verbose['eval_rate'].tolist()
response_time = response_time[:-1]
reset_iters = live[live['Iteration'] == 1].index.tolist()
eval_rates = [np.float64(evals[:-9]) for evals in eval_rate]

In [6]:
live_copy = live.copy()
live_copy = live_copy.dropna()
corr_matrix = live_copy.corr()

In [7]:
live = live.drop(columns=['Memory Current Clock (MHz)'])
live_copy = live_copy.drop(columns=['Memory Current Clock (MHz)', "Memory Clock Utilization"])

In [8]:
live_copy.columns

Index(['GPU Utilization (%)', 'Power Draw (Watts)', 'GPU Temp (°C)',
       'GPU Current Clock (MHz)', 'Memory Allocation Used (MB)',
       'Memory Utilization (%)', 'Current Time', 'Time Delta', 'Iteration',
       'GPU Clock Utilization'],
      dtype='object')

In [9]:
live_copy_of_copy = live_copy.copy()
live_copy_of_copy = live_copy_of_copy.rename(columns={
    "GPU Utilization (%)":"GPU Util",
    "Memory Utilization (%)":"Memory Util",
    'GPU Temp (°C)':"GPU Temp.",
    "GPU Clock Utilization":"GPU Clock Util",
})

live_copy_of_copy = live_copy_of_copy[["GPU Util","Memory Util","GPU Temp.", 'GPU Clock Util']]

In [ ]:
mask = np.triu(np.ones_like(live_copy_of_copy.corr(), dtype=bool))
heatmap = sns.heatmap(live_copy_of_copy.corr(), cmap='crest', annot=True)

In [11]:
# tri_up = corr_matrix.mask(mask)
# to_drop = [col for col in tri_up.columns if any(tri_up[col] > 0.9)]

In [ ]:
pair_plot = sns.pairplot(live_copy.iloc[:, 3:-3])

In [11]:
column_names = live.columns.tolist()
live_avg = dict(zip(column_names, [list(range(408))] * len(column_names)))
live_avg = pd.DataFrame(live_avg, dtype=np.float64)

In [12]:
temp_df = dict()
count = 0
for iter in range(1, len(reset_iters)):
    # print(f"Count: {str(count)}")
    start_index = reset_iters[iter-1]
    end_index = reset_iters[iter]-1
    # print(f"Current Slice: {str(start_index)}:{str(end_index)}")
    for col in range(len(column_names)):
        temp_list = live.iloc[start_index:end_index,  col].tolist()
        # print(f"On column: {str(col)} with the average: {str(np.average(temp_list))}")
        live_avg.iloc[count, col] = np.float64(np.average(temp_list))
    count+=1

In [13]:
live_avg = live_avg.iloc[:-1,:]

In [15]:
from sklearn.preprocessing import normalize, StandardScaler

live_avg_norm = normalize(live_avg)
ss = StandardScaler()
live_avg_scaled = ss.fit_transform(live_avg)

In [16]:
live_avg_scaled = pd.DataFrame(data=live_avg_scaled)
live_avg_scaled.columns = live_avg.columns


live_avg_norm = pd.DataFrame(data=live_avg_norm)
# live_avg_norm['Response Time'] = live_avg.iloc[: , -2:-1]
live_avg_norm.columns = live_avg_norm.columns

In [17]:
live_avg['Resonse Time'] = response_time
live_avg['Eval Rate'] = eval_rates[:-1]

In [18]:
live_avg.to_csv("../data/samples/mega_processed/live_avg.csv")
live_avg_scaled.to_csv("../data/samples/mega_processed/live_avg_scaled.csv")
live_avg_norm.to_csv("../data/samples/mega_processed/live_avg_norm.csv")